# Case Study: Position Control of a Toy Car

**Learning Goals**
- Model a physical system and derive its transfer function
- See why open-loop position control fails for a double-integrator plant
- Compare P, PD, and PID controllers for position control
- Calculate steady-state error using the final value theorem
- Understand how each controller term shapes the closed-loop response

## Relevant lecture videos

In [1]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://echo360.org/media/b9e01644-b429-4638-b858-bc6b519ad344/public?autoplay=false&automute=false&currentMediaId=433fd0ae-0c8e-4722-9cac-bf4173d53eb3" frameborder="0" allowfullscreen></iframe>')

/home/matvei/JupyterBasedControlEngineeringTextbook/venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [2]:
HTML('<iframe width="560" height="315" src="https://echo360.org/media/daa3d523-91f8-4f2c-83bb-9992a6317539/public?autoplay=false&automute=false&currentMediaId=245808a1-4e74-47cb-97ea-818f8b7b01da&sessionId=16b7914b-9ec9-4500-8ec2-07c79892fbc9" frameborder="0" allowfullscreen></iframe>')

In [3]:
HTML('<iframe width="560" height="315" src="https://echo360.org/media/c59ac0df-22cf-4ef3-9b13-7bff9bd27d48/public?autoplay=false&automute=false&currentMediaId=2667f022-8b89-4d84-8bbb-3323fb9b0348" frameborder="0" allowfullscreen></iframe>')

In [4]:
HTML('<iframe width="560" height="315" src="https://echo360.org/media/ed723125-bf8b-4ad6-a1a1-67e4d666d144/public?autoplay=false&automute=false&currentMediaId=6a93ac61-95ab-426e-8bad-0168059666db" frameborder="0" allowfullscreen></iframe>')

In [5]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

print("Libraries loaded.")

Libraries loaded.


In [7]:
def closed_loop_step(Kp=1.0, Kd=0.0, Ki=0.0, m=1.0, t_max=20.0):
    """Simulate closed-loop step response of toy car with given controller gains."""
    t = np.linspace(0, t_max, 2000)
    # Plant: G(s) = 1/(m*s^2)
    # Controller: C(s) = Kp + Kd*s + Ki/s
    # Closed-loop: Y(s)/R(s) = C(s)G(s)/(1 + C(s)G(s))
    # = (Kd*s^2 + Kp*s + Ki) / (m*s^3 + Kd*s^2 + Kp*s + Ki)
    num = [Kd, Kp, Ki]
    den = [m, Kd, Kp, Ki]
    if Ki == 0 and Kd == 0:
        # P-only: num = [Kp], den = [m, 0, Kp]
        num = [Kp]
        den = [m, 0, Kp]
    elif Ki == 0:
        # PD: num = [Kd, Kp], den = [m, Kd, Kp]
        num = [Kd, Kp]
        den = [m, Kd, Kp]
    sys = signal.TransferFunction(num, den)
    _, y = signal.step(sys, T=t)
    return t, y

def plot_response(t, y, title, poles, setpoint=1.0):
    """Plot time response and pole-zero map."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(t, y, 'b-', lw=2)
    ax1.axhline(setpoint, color='gray', ls='--', alpha=0.5, label='Setpoint')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Position (m)')
    ax1.set_title(title)
    ax1.set_xlim(0, t[-1])
    ax1.grid(alpha=0.3)
    ax1.legend(fontsize=9)

    limit = max(5, np.ceil(max(abs(p.real) for p in poles)) + 1,
                np.ceil(max(abs(p.imag) for p in poles)) + 1) if len(poles) > 0 else 5
    ax2.axhline(0, color='k', linewidth=0.8)
    ax2.axvline(0, color='k', linewidth=0.8)
    ax2.set_xlim(-limit, limit)
    ax2.set_ylim(-limit, limit)
    ax2.set(xlabel='Re($s$)', ylabel='Im($s$)', title='Pole Locations')
    ax2.grid(alpha=0.3)
    for p in poles:
        ax2.plot(p.real, p.imag, 'rx', markersize=12, markeredgewidth=3)
    ax2.axvspan(-limit, 0, alpha=0.06, color='g')
    ax2.axvspan(0, limit, alpha=0.06, color='r')
    ax2.text(-limit*0.9, limit*0.85, 'Stable', fontsize=9, color='g', fontweight='bold')
    ax2.text(limit*0.05, limit*0.85, 'Unstable', fontsize=9, color='r', fontweight='bold')
    ax2.axis('equal')

    fig.tight_layout()
    plt.show()

    pole_str = ', '.join([f'{p.real:.3f} {p.imag:+.3f}j' for p in poles])
    print(f'Closed-loop poles: {pole_str}')

print("Helpers defined.")

Helpers defined.


In [8]:
def make_animation(t, x, setpoint=None, title=''):
    fig = plt.figure(figsize=(10, 5))
    ax_top = fig.add_subplot(2, 1, 1)
    ax_bot = fig.add_subplot(2, 1, 2)

    x_max = max(abs(x)) * 1.4 if max(abs(x)) > 0 else 15
    if setpoint is not None:
        x_max = max(x_max, setpoint * 1.4)
    ax_top.set_xlim(-1, x_max + 2)
    ax_top.set_ylim(-0.5, 3.5)
    ax_top.set_aspect('equal')
    ax_top.set_ylabel('position')
    ax_top.set_title(title, fontsize=11)
    ax_top.grid(alpha=0.3)
    ax_top.axhline(0.2, color='gray', lw=3)

    if setpoint is not None:
        ax_top.axvline(setpoint, color='green', ls='--', alpha=0.5, label=f'target = {setpoint} m')
        ax_top.legend(fontsize=8, loc='upper left')

    from matplotlib.patches import Rectangle, Circle
    from matplotlib.lines import Line2D
    car = Rectangle((0, 0.5), 1.5, 1.0, fc='royalblue', ec='k', lw=2)
    ax_top.add_patch(car)

    rw = 0.2
    w1_body = Circle((0, 0), rw, fc='k', ec='k')
    w2_body = Circle((0, 0), rw, fc='k', ec='k')
    ax_top.add_patch(w1_body)
    ax_top.add_patch(w2_body)
    spoke1 = Line2D([0, 0], [0, 0], color='white', lw=2)
    spoke2 = Line2D([0, 0], [0, 0], color='white', lw=2)
    ax_top.add_line(spoke1)
    ax_top.add_line(spoke2)

    time_text = ax_top.text(0.02, 0.9, '', transform=ax_top.transAxes, fontsize=9, fontfamily='monospace')
    pos_text = ax_top.text(0.02, 0.78, '', transform=ax_top.transAxes, fontsize=9, fontfamily='monospace', color='royalblue')

    ax_bot.plot(t, x, 'b-', lw=1, alpha=0.3)
    if setpoint is not None:
        ax_bot.axhline(setpoint, color='green', ls='--', alpha=0.5, label=f'target {setpoint} m')
    ax_bot.set_xlim(0, t[-1])
    y_lim = max(abs(x)) * 1.2 if max(abs(x)) > 0 else 10
    ax_bot.set_ylim(-y_lim * 0.1, y_lim)
    ax_bot.set_xlabel('time (s)')
    ax_bot.set_ylabel('position (m)')
    ax_bot.grid(alpha=0.3)
    ax_bot.legend(fontsize=8)
    trace, = ax_bot.plot([], [], 'b-', lw=2)

    fig.tight_layout()

    n_frames = 60
    step = max(1, len(t) // n_frames)

    off1, off2 = 0.3, 1.2

    def init():
        car.set_x(0)
        w1_body.center = (off1, 0.5)
        w2_body.center = (off2, 0.5)
        spoke1.set_data([off1, off1 + rw], [0.5, 0.5])
        spoke2.set_data([off2, off2 + rw], [0.5, 0.5])
        trace.set_data([], [])
        time_text.set_text('')
        pos_text.set_text('')
        return car, w1_body, w2_body, spoke1, spoke2, trace, time_text, pos_text

    def animate(i):
        idx = min(i * step, len(t) - 1)
        px = x[idx]
        car.set_x(px)
        cx1, cx2 = px + off1, px + off2
        cy = 0.5
        w1_body.center = (cx1, cy)
        w2_body.center = (cx2, cy)
        angle = px / rw
        spoke1.set_data([cx1, cx1 - rw * np.cos(-angle)],
                        [cy, cy - rw * np.sin(-angle)])
        spoke2.set_data([cx2, cx2 - rw * np.cos(-angle)],
                        [cy, cy - rw * np.sin(-angle)])
        trace.set_data(t[:idx], x[:idx])
        time_text.set_text(f'time = {t[idx]:.1f} s')
        pos_text.set_text(f'position = {px:.1f} m')
        return car, w1_body, w2_body, spoke1, spoke2, trace, time_text, pos_text

    ani = FuncAnimation(fig, animate, frames=n_frames,
                        init_func=init, blit=True, interval=80)
    plt.close(fig)
    return HTML(ani.to_jshtml())

print("Animation helper ready.")

Animation helper ready.


---

## System Model

Consider a toy car of mass $m$ on a frictionless surface. Newton's second law gives:

$$F(t) = m \, a(t) = m \, \ddot{x}(t)$$

where $F(t)$ is the applied force (input) and $x(t)$ is the position (output).

Taking the Laplace transform with zero initial conditions:

$$F(s) = m \, s^2 X(s) \quad\Longrightarrow\quad G(s) = \frac{X(s)}{F(s)} = \frac{1}{m\,s^2}$$

This is a **double integrator** with two poles at $s = 0$. The system has no natural damping or restoring force.

---

## Open-Loop Response

Apply a constant force $F = 1$ N (step input). The open-loop step response is $x(t) = \frac{1}{2m}t^2$ (a parabola). The position grows without bound, meaning the car never settles at a desired position.

There is no way to tell the car *where* to go in open loop; we can only push it.

In [9]:
m_slider = widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0,
                               description='Mass $m$ (kg):',
                               style={'description_width': 'initial'})
ol_btn = widgets.Button(description='Run', button_style='primary')
ol_out = widgets.Output()

def on_ol_run(b):
    with ol_out:
        clear_output(wait=True)
        m = m_slider.value
        t = np.linspace(0, 10, 1000)
        # Open-loop step response: x(t) = (1/2m) * t^2
        x = 0.5 / m * t**2
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(t, x, 'b-', lw=2)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Position (m)')
        ax.set_title(f'Open-Loop Step Response ($m = {m:.1f}$ kg)')
        ax.grid(alpha=0.3)
        fig.tight_layout()
        plt.show()
        print('Position grows without bound — the car never reaches a steady position.')

ol_btn.on_click(on_ol_run)
display(widgets.VBox([m_slider, ol_btn, ol_out]))

print("Adjust the mass and click Run.")

Adjust the mass and click Run.


**Observation.** The open-loop toy car is a pure double integrator. With two poles at $s = 0$, the system is on the stability boundary. Constant force leads to unbounded position, so we need **feedback control**.

---

## P Controller

A **proportional (P) controller** applies force proportional to the error:

$$F(t) = K_p\,\bigl(r(t) - x(t)\bigr)$$

where $r(t)$ is the reference (desired position) and $K_p$ is the proportional gain.

Closed-loop transfer function:

$$\frac{X(s)}{R(s)} = \frac{K_p}{m\,s^2 + K_p}$$

The poles are at $s = \pm j\sqrt{K_p/m}$ (on the imaginary axis). This produces **sustained oscillations**.

In [10]:
Kp_p_slider = widgets.FloatSlider(min=0.1, max=10.0, step=0.1, value=2.0,
                                  description='$K_p$:',
                                  style={'description_width': 'initial'})
p_btn = widgets.Button(description='Run', button_style='primary')
p_out = widgets.Output()

def on_p_run(b):
    with p_out:
        clear_output(wait=True)
        Kp = round(Kp_p_slider.value, 1)
        m = 1.0
        t, y = closed_loop_step(Kp=Kp, m=m, t_max=20.0)
        poles = [complex(0, np.sqrt(Kp/m)), complex(0, -np.sqrt(Kp/m))]
        display(make_animation(t, y, setpoint=1.0, title=f'P Control ($K_p = {Kp:.1f}$)'))
        plot_response(t, y, f'P Control ($K_p = {Kp:.1f}$)', poles)

p_btn.on_click(on_p_run)
display(widgets.VBox([Kp_p_slider, p_btn, p_out]))

print("Adjust Kp and click Run.")

Adjust Kp and click Run.


**Observation.** With P control only, the poles lie on the imaginary axis. The response is a pure sinusoid that never decays, and the car oscillates forever around the setpoint. Increasing $K_p$ makes the oscillations faster.

---

## PD Controller

Adding **derivative** action gives a PD controller:

$$F(t) = K_p\,\bigl(r(t) - x(t)\bigr) + K_d\,\bigl(\dot{r}(t) - \dot{x}(t)\bigr)$$

The derivative term acts like a **damper**, as it opposes velocity.

Closed-loop transfer function:

$$\frac{X(s)}{R(s)} = \frac{K_d\,s + K_p}{m\,s^2 + K_d\,s + K_p}$$

The characteristic equation is $m\,s^2 + K_d\,s + K_p = 0$. By choosing $K_d$, we can place the poles anywhere in the left half-plane.

In [11]:
Kp_pd_slider = widgets.FloatSlider(min=0.1, max=10.0, step=0.1, value=4.0,
                                   description='$K_p$:',
                                   style={'description_width': 'initial'})
Kd_slider = widgets.FloatSlider(min=0.0, max=10.0, step=0.1, value=4.0,
                                description='$K_d$:',
                                style={'description_width': 'initial'})
pd_btn = widgets.Button(description='Run', button_style='primary')
pd_out = widgets.Output()

def on_pd_run(b):
    with pd_out:
        clear_output(wait=True)
        Kp = round(Kp_pd_slider.value, 1)
        Kd = round(Kd_slider.value, 1)
        m = 1.0
        t, y = closed_loop_step(Kp=Kp, Kd=Kd, m=m, t_max=10.0)
        # Poles of ms^2 + Kd*s + Kp = 0
        disc = Kd**2 - 4*m*Kp
        if disc >= 0:
            poles = [(-Kd + np.sqrt(disc)) / (2*m), (-Kd - np.sqrt(disc)) / (2*m)]
        else:
            poles = [complex(-Kd/(2*m), np.sqrt(-disc)/(2*m)),
                     complex(-Kd/(2*m), -np.sqrt(-disc)/(2*m))]
        display(make_animation(t, y, setpoint=1.0, title=f'PD Control ($K_p = {Kp:.1f},\; K_d = {Kd:.1f}$)'))
        plot_response(t, y, f'PD Control ($K_p = {Kp:.1f},\; K_d = {Kd:.1f}$)', poles)

pd_btn.on_click(on_pd_run)
display(widgets.VBox([Kp_pd_slider, Kd_slider, pd_btn, pd_out]))

print("Adjust Kp and Kd, then click Run.")

<>:24: SyntaxWarning: invalid escape sequence '\;'
<>:25: SyntaxWarning: invalid escape sequence '\;'
<>:24: SyntaxWarning: invalid escape sequence '\;'
<>:25: SyntaxWarning: invalid escape sequence '\;'
/tmp/ipykernel_118626/1921362297.py:24: SyntaxWarning: invalid escape sequence '\;'
  display(make_animation(t, y, setpoint=1.0, title=f'PD Control ($K_p = {Kp:.1f},\; K_d = {Kd:.1f}$)'))
/tmp/ipykernel_118626/1921362297.py:25: SyntaxWarning: invalid escape sequence '\;'
  plot_response(t, y, f'PD Control ($K_p = {Kp:.1f},\; K_d = {Kd:.1f}$)', poles)


Adjust Kp and Kd, then click Run.


**Observation.** The derivative term $K_d$ provides damping. Poles now have negative real parts, so oscillations decay. Higher $K_d$ gives more damping (less overshoot); higher $K_p$ gives faster response.

---

## PID Controller

Adding **integral** action gives a PID controller:

$$F(t) = K_p\,e(t) + K_i\int_0^t e(\tau)\,d\tau + K_d\,\dot{e}(t)$$

where $e(t) = r(t) - x(t)$. The integral term accumulates past errors, ensuring zero steady-state error even under disturbances.

Closed-loop transfer function:

$$\frac{X(s)}{R(s)} = \frac{K_d\,s^2 + K_p\,s + K_i}{m\,s^3 + K_d\,s^2 + K_p\,s + K_i}$$

In [12]:
Kp_pid_slider = widgets.FloatSlider(min=0.1, max=10.0, step=0.1, value=4.0,
                                    description='$K_p$:',
                                    style={'description_width': 'initial'})
Ki_slider = widgets.FloatSlider(min=0.0, max=5.0, step=0.1, value=0.5,
                                description='$K_i$:',
                                style={'description_width': 'initial'})
Kd_pid_slider = widgets.FloatSlider(min=0.0, max=10.0, step=0.1, value=4.0,
                                    description='$K_d$:',
                                    style={'description_width': 'initial'})
pid_btn = widgets.Button(description='Run', button_style='primary')
pid_out = widgets.Output()

def on_pid_run(b):
    with pid_out:
        clear_output(wait=True)
        Kp = round(Kp_pid_slider.value, 1)
        Ki = round(Ki_slider.value, 1)
        Kd = round(Kd_pid_slider.value, 1)
        m = 1.0
        t, y = closed_loop_step(Kp=Kp, Ki=Ki, Kd=Kd, m=m, t_max=10.0)
        # Poles of ms^3 + Kd*s^2 + Kp*s + Ki = 0
        poles = np.roots([m, Kd, Kp, Ki])
        display(make_animation(t, y, setpoint=1.0, title=f'PID Control ($K_p = {Kp:.1f},\; K_i = {Ki:.1f},\; K_d = {Kd:.1f}$)'))
        plot_response(t, y, f'PID Control ($K_p = {Kp:.1f},\; K_i = {Ki:.1f},\; K_d = {Kd:.1f}$)', poles)

pid_btn.on_click(on_pid_run)
display(widgets.VBox([Kp_pid_slider, Ki_slider, Kd_pid_slider, pid_btn, pid_out]))

print("Adjust Kp, Ki, Kd, then click Run.")

<>:23: SyntaxWarning: invalid escape sequence '\;'
<>:23: SyntaxWarning: invalid escape sequence '\;'
<>:24: SyntaxWarning: invalid escape sequence '\;'
<>:24: SyntaxWarning: invalid escape sequence '\;'
<>:23: SyntaxWarning: invalid escape sequence '\;'
<>:23: SyntaxWarning: invalid escape sequence '\;'
<>:24: SyntaxWarning: invalid escape sequence '\;'
<>:24: SyntaxWarning: invalid escape sequence '\;'
/tmp/ipykernel_118626/2582244997.py:23: SyntaxWarning: invalid escape sequence '\;'
  display(make_animation(t, y, setpoint=1.0, title=f'PID Control ($K_p = {Kp:.1f},\; K_i = {Ki:.1f},\; K_d = {Kd:.1f}$)'))
/tmp/ipykernel_118626/2582244997.py:23: SyntaxWarning: invalid escape sequence '\;'
  display(make_animation(t, y, setpoint=1.0, title=f'PID Control ($K_p = {Kp:.1f},\; K_i = {Ki:.1f},\; K_d = {Kd:.1f}$)'))
/tmp/ipykernel_118626/2582244997.py:24: SyntaxWarning: invalid escape sequence '\;'
  plot_response(t, y, f'PID Control ($K_p = {Kp:.1f},\; K_i = {Ki:.1f},\; K_d = {Kd:.1f}$)', p

Adjust Kp, Ki, Kd, then click Run.


---

## Steady-State Error

The **steady-state error** $e_{ss}$ is the difference between the reference and the output as $t \to \infty$. We compute it using the **final value theorem**:

$$e_{ss} = \lim_{t \to \infty} e(t) = \lim_{s \to 0} s\,E(s)$$

where $E(s) = R(s) - Y(s) = \dfrac{R(s)}{1 + C(s)G(s)}$.

**Important:** the final value theorem only gives a valid result when all poles of $s\,E(s)$ lie strictly in the left half-plane. If any pole is on the imaginary axis (or in the right half-plane), the system never reaches a steady state and the theorem cannot be applied.

For a **step input** $R(s) = 1/s$:

$$E(s) = \frac{1/s}{1 + C(s)G(s)} \quad\Longrightarrow\quad
s\,E(s) = \frac{1}{1 + C(s)G(s)}$$

### P Controller ($C(s) = K_p$)

The closed-loop poles are at $s = \pm j\sqrt{K_p/m}$ and are purely imaginary with no real part. These are poles of $E(s)$ as well, so the final value theorem **does not apply**. The system never settles: it oscillates forever around the setpoint. There is no meaningful steady-state error.

### PD Controller ($C(s) = K_p + K_d\,s$)

For $K_d > 0$, the closed-loop poles $s = \frac{-K_d \pm \sqrt{K_d^2 - 4mK_p}}{2m}$ are in the left half-plane. The final value theorem applies:

$$e_{ss} = \lim_{s \to 0} \frac{1}{1 + \frac{K_d\,s + K_p}{m\,s^2}}
= \lim_{s \to 0} \frac{m\,s^2}{m\,s^2 + K_d\,s + K_p} = 0$$

The two integrators in $G(s)$ ensure zero steady-state error to a step.

### PID Controller ($C(s) = K_p + K_i/s + K_d\,s$)

With $K_d > 0$ and appropriate gains, all closed-loop poles are in the left half-plane. The final value theorem gives:

$$e_{ss} = \lim_{s \to 0} \frac{1}{1 + \frac{K_d\,s + K_p + K_i/s}{m\,s^2}}
= \lim_{s \to 0} \frac{m\,s^2}{m\,s^2 + K_d\,s + K_p + K_i/s} = 0$$

The table below summarises:

| Controller | Poles | Transient behaviour | Steady-state error (step) |
|---|---|---|---|
| **P** | $\pm j\sqrt{K_p/m}$ (imaginary axis) | Sustained oscillation | N/A (never settles) |
| **PD** | $\frac{-K_d \pm \sqrt{K_d^2 - 4mK_p}}{2m}$ (LHP) | Damped, adjustable | 0 |
| **PID** | Roots of $m s^3 + K_d s^2 + K_p s + K_i$ (LHP) | Damped + integral action | 0 |

The integral term in PID is particularly beneficial for **rejecting constant disturbances**. A a steady disturbance force will cause a PD-controlled system to drift, while PID will eliminate the error.

---

## Summary

- The toy car is a **double integrator** $G(s) = 1/(ms^2)$, so open-loop position is uncontrollable.
- **P control** adds a restoring force but leaves poles on the imaginary axis → sustained oscillations.
- **PD control** adds damping through the derivative term → poles can be placed in the left half-plane → settling response.
- **PID control** adds integral action for disturbance rejection.
- The **final value theorem** $e_{ss} = \lim_{s \to 0} s\,E(s)$ computes steady-state error, but only applies when all poles of $s\,E(s)$ are in the left half-plane.
- **P control** never settles (poles on imaginary axis), so the final value theorem does not apply.
- **PD** and **PID control** achieve $e_{ss} = 0$ for a step input (the plant has two integrators). The choice depends on the desired transient behaviour and disturbance rejection.